# Notebook 1: The Optimization Engine

The goal of this project is to use Monte Carlo statistical optimization to search a 10-dimensional hyperspace for the "True Champion" of the *battleMage* arena.

The search space for this 10-parameter strategy model is vast, and the "hyper-island" containing the competent bots is vanishingly small. A brute-force search is computationally unfeasible. To proceed, I engineered a Multi-Stage Statistical Optimization Process to progressively narrow the search space, and I devised a gating system to minimize computational time spent on statistically subpar bots. In total, this system executed the simulation of hundreds of billions of battles to isolate the optimal strategy.

This notebook documents the engineering process:
*  **System Calibration (Phase 0):** Normalizing input features and quantifying the game's inherent noise.
*  **Exploration (Phase 1):** Mapping the 10D space to identify irrelevant features and broad "hot zones."
*  **The Engineering Pivot:** Designing a "Dynamic $N$ Gating System" to solve the computational bottleneck.
*  **Exploitation (Phase 2):** Deploying the gated engine on a search space reduced by **99.999908%**, isolating a champion strategy that is **98 Standard Errors** statistically superior to the Rule-Based Benchmark.

## Data Provenance

This project utilizes a Java-based simulation engine I created to generate synthetic data. I built the battleMage system as a platform to teach strategic program design and algorithm development, which also enables the generation of vast quantities of clean, high-quality gameplay data.

This notebook analyzes precomputed simulation results generated by a multi-phase optimization process. To run the notebook, you must download the dataset from **GitHub Releases** and extract it into the `data/` directory. See [***data README***](/data/README.md) for details.



## battleMage System Dynamics

* **The "Arena Challenge" Goal:**  Survive as many rounds as possible, against opponents of increasing difficulty level.

* **The Resources**, 200 points to allocate:

    * **Hit Points (HP):** Your life. Also spent to perform actions.

    * **Stamina:** Needed to block.

* **The Threat:** Each turn, damage is rolled for 4 "Quadrants" (Q1-Q4). Damage is removed from your HP if not mitigated.

* Each turn, you choose an **action** to respond to the threat:

<div style="text-align: left;">


### The Actions:

<table style="width: auto; margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align:left; padding: 8px;">Action</th>
      <th style="text-align:left; padding: 8px;">Strategic Result</th>
      <th style="text-align:left; padding: 8px;">Resource Used</th>
      <th style="text-align:left; padding: 8px;">Cost Formula</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align:left; padding: 8px;"><strong>Attack ($N$ Hits)</strong></td>
      <td style="text-align:left; padding: 8px;">Reduces Opponent HP by $10N$.</td>
      <td style="text-align:left; padding: 8px;">Any</td>
      <td style="text-align:left; padding: 8px;">$3^N - 2$</td>
    </tr>
    <tr>
      <td style="text-align:left; padding: 8px;"><strong>Block</strong></td>
      <td style="text-align:left; padding: 8px;">Removes incoming threat from 2 adjacent quadrants.</td>
      <td style="text-align:left; padding: 8px;">Stamina</td>
      <td style="text-align:left; padding: 8px;">$1$</td>
    </tr>
    <tr>
      <td style="text-align:left; padding: 8px;"><strong>Magic Blast</strong></td>
      <td style="text-align:left; padding: 8px;">Reduces Opponent HP by $1/3$.</td>
      <td style="text-align:left; padding: 8px;">Any</td>
      <td style="text-align:left; padding: 8px;">$10 - \text{Number Used}$</td>
    </tr>
    <tr>
      <td style="text-align:left; padding: 8px;"><strong>Magic Shield ($N$ Power)</strong></td>
      <td style="text-align:left; padding: 8px;">Reduces damage from all 4 quadrants by half, $N$ times.</td>
      <td style="text-align:left; padding: 8px;">Any</td>
      <td style="text-align:left; padding: 8px;">$2^N$</td>
    </tr>
  </tbody>
</table>

</div>

## Rationale

My initial solution for the game was a heuristic decision maker which maximized the difference between the opponent's fractional loss of hit points and the player's. However, one student's hand-coded strategy ('The Rules-based Benchmark') slightly but significantly outperformed this solution.

To surpass this benchmark, I introduced a set of **10 tunable parameters** into my solution's decision logic. **The goal of this project is to use statistical optimization to search 10-dimensional hyperspace for the True Champion of battleMage!**

## Optimization Parameters

<div style="text-align: left;">

<table style="width: auto; margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align:left; padding: 8px;">Parameter Category</th>
      <th style="text-align:left; padding: 8px;">Variable</th>
      <th style="text-align:left; padding: 8px;">Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align:left; padding: 8px;"><strong>Resource Management</strong></td>
      <td style="text-align:left; padding: 8px;"><code>w_alloc</code></td>
      <td style="text-align:left; padding: 8px;">Percentage of HP pool allocated to Stamina at start.</td>
    </tr>
    <tr>
      <td style="text-align:left; padding: 8px;"></td>
      <td style="text-align:left; padding: 8px;"><code>w_cost</code></td>
      <td style="text-align:left; padding: 8px;">Penalty for resource cost (found irrelevant in phase 1 and removed).</td>
    </tr>
    <tr>
      <td style="text-align:left; padding: 8px;"><strong>Dynamic Scoring</strong></td>
      <td style="text-align:left; padding: 8px;"><code>w_ratioGain</code></td>
      <td style="text-align:left; padding: 8px;">Defensiveness: Penalizes HP loss, weighted more heavily when the opponent is healthy.</td>
    </tr>
    <tr>
      <td style="text-align:left; padding: 8px;"></td>
      <td style="text-align:left; padding: 8px;"><code>w_ratioLoss</code></td>
      <td style="text-align:left; padding: 8px;">Aggressiveness: Rewards dealing damage, weighted more heavily when the player is healthy.</td>
    </tr>
    <tr>
      <td style="text-align:left; padding: 8px;"><strong>Static Scoring</strong></td>
      <td style="text-align:left; padding: 8px;"><code>w_playerHPdelta</code></td>
      <td style="text-align:left; padding: 8px;">Weight for raw damage taken/prevented (Player HP).</td>
    </tr>
    <tr>
      <td style="text-align:left; padding: 8px;"></td>
      <td style="text-align:left; padding: 8px;"><code>w_oppHPdelta</code></td>
      <td style="text-align:left; padding: 8px;">Weight for raw damage dealt (Opponent HP).</td>
    </tr>
    <tr>
      <td style="text-align:left; padding: 8px;"><strong>Action Biases</strong></td>
      <td style="text-align:left; padding: 8px;"><code>w_attackBias</code></td>
      <td style="text-align:left; padding: 8px;">Score bonus/penalty for Attack actions.</td>
    </tr>
    <tr>
      <td style="text-align:left; padding: 8px;"></td>
      <td style="text-align:left; padding: 8px;"><code>w_blockBias</code></td>
      <td style="text-align:left; padding: 8px;">Score bonus/penalty for Block actions.</td>
    </tr>
    <tr>
      <td style="text-align:left; padding: 8px;"></td>
      <td style="text-align:left; padding: 8px;"><code>w_blastBias</code></td>
      <td style="text-align:left; padding: 8px;">Score bonus/penalty for Magic Blast actions.</td>
    </tr>
    <tr>
      <td style="text-align:left; padding: 8px;"></td>
      <td style="text-align:left; padding: 8px;"><code>w_shieldBias</code></td>
      <td style="text-align:left; padding: 8px;">Score bonus/penalty for Magic Shield actions.</td>
    </tr>

  </tbody>
</table>

</div>

## Project Setup - Imports and Helper Functions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
from matplotlib.ticker import FuncFormatter, MultipleLocator
from typing import Dict, List, Tuple, Any
import matplotlib.ticker as mtick
from pathlib import Path


#ensure an images folder exists
Path("images").mkdir(exist_ok=True)

#ensure data files are present.
DATA_DIR = Path("data")

required_files = [
    "phase_zero_outcomes.csv",
    "phase_zero_actions.csv",
    "phase_one_outcomes.csv",
    "phase_one_refined_outcomes.csv",
    "phase_two_outcomes.csv",
    "phase_two_refined_outcomes.csv",
    "noise_calibration.csv"
]

missing = [f for f in required_files if not (DATA_DIR / f).exists()]

if missing:
    raise FileNotFoundError(
        "Required data files are missing.\n\n"
        "This notebook analyzes precomputed simulation results.\n"
        "Please download the full dataset from GitHub Releases and extract it into the `data/` directory.\n\n"
        f"Missing files:\n- " + "\n- ".join(missing)
    )
else:
    print("Loading precomputed battleMage simulation data.")
    

# Set style for the plots
sns.set(style="whitegrid")

# Define the structure for the explore_ranges dictionary (Phase 1 baseline)
# Using the same order as in the Java OptimizingChampion constructor.
INITIAL_EXPLORE_RANGES: Dict[str, Tuple[float, float]] = {
    'w_alloc': (0.0, 1.0),
    'w_cost': (0.0, 1.0),
    'w_ratioGain': (0.0, 1.0),
    'w_ratioLoss': (0.0, 1.0),
    'w_playerHPdelta': (0.0, 1.0),
    'w_oppHPdelta': (0.0, 1.0),
    'w_attackBias': (-1.0, 1.0),
    'w_blockBias': (-1.0, 1.0),
    'w_blastBias': (-1.0, 1.0),
    'w_shieldBias': (-1.0, 1.0)
}

# Define the 10 parameters for looping consistency
ALL_PARAMETERS: List[str] = list(INITIAL_EXPLORE_RANGES.keys())

# Initialize the list to store results for the final progress table
progress_report = [] 
# Initialize the Rules-based benchmark. Creating a bot that statistically surpasses this
# is the goal of the project
RULES_BASED_BENCHMARK = 25.10

# --- Function 1: Load Data and Generate Histogram ---

def load_data_and_generate_histogram(file_path: str, phase_name: str, quantile_level: float):
    """
    Loads data, calculates the elite cutoff, and generates the overall performance histogram.

    Returns the loaded DataFrame (df) and the filtered DataFrame (top_runs) for the next step.
    """
    print(f"Loading data for {phase_name} from {file_path}...")
    
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"ERROR: File not found at {file_path}. Cannot proceed.")
        return None, None

    # Calculate Elite Runs
    top_quantile_value = df['Avg_Final_Level'].quantile(quantile_level)
    top_runs = df[df['Avg_Final_Level'] >= top_quantile_value]
    
    # Calculate visible percentage (e.g., 0.9995 -> 0.05%)
    visible_percent = (1 - quantile_level) * 100 
    
    print(f"\nSuccessfully loaded {len(df)} simulation runs.")
    print(f"Top {visible_percent:.3f}% Cutoff: {top_quantile_value:.2f} (Samples: {len(top_runs)})")

    # --- Generate Histogram Plot ---
    plt.figure(figsize=(10, 6))
    sns.histplot(df['Avg_Final_Level'], kde=True, bins=50) 
    
    plt.title(f"{phase_name}: Distribution of Avg. Final Level | {len(df)} Runs", fontsize=16, fontweight='bold')
    plt.xlabel("Average Final Level Achieved")
    plt.ylabel("Count of Parameter Sets")

    plt.axvline(top_quantile_value, color='red', linestyle='--', 
                label=f'Top {visible_percent:.3f}% | Cutoff Level {top_quantile_value:.2f}')
    plt.legend()
    plt.savefig(f"images/{phase_name.replace(' ', '_')}_histogram.png", bbox_inches='tight')
    plt.show()

    return df, top_runs


# --- Function 2: Create Parameter Distribution Plots ---

def create_distribution_plots(
    df: pd.DataFrame, 
    top_runs: pd.DataFrame, 
    phase_name: str, 
    quantile_level: float, 
    explore_ranges: Dict[str, Tuple[float, float]]
):
    """
    Generates the grid of overlaid distribution plots for only the active parameters
    (skipping any parameter where the range size is 0.0).
    """
    print(f"\nGenerating parameter distribution plots for {phase_name}...")
    
    visible_percent = (1 - quantile_level) * 100
    
    # 1. IDENTIFY ACTIVE PARAMETERS BY CHECKING RANGE SIZE
    active_parameters = []
    for param in ALL_PARAMETERS:
        original_min, original_max = explore_ranges.get(param, (-1.0, 1.0))
        range_size = original_max - original_min
        
        # Only plot if the range size is greater than a tiny epsilon (to account for floating point)
        if range_size > 0.0001: 
            active_parameters.append(param)
            
    # Calculate the grid size based on the *remaining* active parameters
    num_plots = len(active_parameters)
    rows = (num_plots + 2) // 3 
    
    plt.figure(figsize=(15, 4 * rows))
    plt.suptitle(f"{phase_name}: Parameter Distributions (All vs. Top {visible_percent:.3f}%)", 
                 fontsize=18, y=1.03)

    for i, param in enumerate(active_parameters): # <-- Iterate over the filtered list
        plt.subplot(rows, 3, i + 1)
        
        # Get the correct limits for plotting
        original_min, original_max = explore_ranges.get(param, (-1.0, 1.0))
        
        # 1. Plot the histogram for ALL runs (Blue)
        sns.histplot(df[param], stat="density", kde=True, color="blue", label="All Runs", alpha=0.3)
        
        # 2. Plot the histogram for TOP runs (Orange)
        sns.histplot(top_runs[param], stat="density", kde=True, color="orange", 
                     label=f"Top {visible_percent:.3f}% Runs", alpha=0.6)

        # 3. Force the plot to show the full defined range
        plt.xlim(original_min, original_max)

        plt.title(f"Distribution of {param}", fontweight='bold')
        plt.xlabel(None)
        plt.ylabel(None)
        plt.legend()

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(f"images/{phase_name.replace(' ', '_')}_distributions.png", bbox_inches='tight')
    plt.show()


# --- Function 3: Calculate New Ranges and Parameter Space Reduction ---

def calculate_new_ranges(
    top_runs: pd.DataFrame, 
    explore_ranges: Dict[str, Tuple[float, float]],
    categories: Dict[str, List[str]]
) -> Dict[str, Tuple[float, float]]:
    """
    Calculates new ranges based on parameter categories (2SD, Ramp Up, etc.).

    Returns the new exploit_ranges_clipped dictionary.
    """
    print("\n--- Calculating New Exploit Ranges ---")
    
    exploit_ranges_clipped: Dict[str, Tuple[float, float]] = {}

    for param in ALL_PARAMETERS:
        
        original_min, original_max = explore_ranges.get(param, (-1.0, 1.0))
        final_low, final_high = original_min, original_max
        
        # NOTE: param_data is only defined for non-Removed parameters
        if param not in categories.get('Removed', []):
            param_data = top_runs[param]
        
        # Default category name for printing
        category_name = "N/A"
        
        # --- 1. Apply Logic Based on Category ---
        
        if param in categories.get('Removed', []):
            final_low, final_high = 0.0, 0.0
            category_name = "REMOVED"
            
        elif param in categories.get('Flat/Complex', []):
            final_low, final_high = original_min, original_max
            category_name = "FLAT/COMPLEX"
            
        elif param in categories.get('Ramp Up', []):
            calc_low = param_data.quantile(0.25)
            final_low = max(original_min, calc_low)
            category_name = "RAMP UP"
            
        elif param in categories.get('Ramp Down', []):
            calc_high = param_data.quantile(0.75)
            final_high = min(original_max, calc_high)
            category_name = "RAMP DOWN"
            
        elif param in categories.get('2SD', []):
            param_mean = param_data.mean()
            param_std = param_data.std()
            calc_low = param_mean - (2 * param_std)
            calc_high = param_mean + (2 * param_std)
            
            final_low = max(original_min, calc_low)
            final_high = min(original_max, calc_high)
            category_name = "2SD"

        # --- 2. Final Clipping and Output (Applied to ALL) ---
        
        # Only check for errors if the parameter hasn't been set to [0, 0]
        if final_low >= final_high and category_name != "REMOVED":
            final_low, final_high = original_min, original_max # Revert if error
        
        exploit_ranges_clipped[param] = (final_low, final_high)
 
        
  
        print(f"  {param} ({category_name}): [{final_low:.3f}, {final_high:.3f}]")

    print("--- New Exploit Ranges Calculated ---\n")
    return exploit_ranges_clipped

#****function 4******
# determine the score with 95% Cofidence interval of the phase-best bot
# check to see if benchmark is beaten significantly
def progress_update(df, phase_name, n_samples):
    """
    Finds the best bot, calculates margin of error using GLOBAL_SIGMA, 
    and determines statistical significance vs Rules-based Benchmark.
    """
    best_run_idx = df['Avg_Final_Level'].idxmax()
    best_score = df.loc[best_run_idx, 'Avg_Final_Level']
    
    # 1. Calculate Statistics
    # Standard Error is the noise of the *average*, which shrinks with N
    standard_error = GLOBAL_SIGMA / np.sqrt(n_samples)
    
    # 95% Confidence Interval (1.96 * SE)
    margin_of_error = 1.96 * standard_error
    
    # The "Z-Score": How many Standard Errors is this above benchmark?
    gap = best_score - RULES_BASED_BENCHMARK
    z_score = gap / standard_error
    
    # 2. Print the Report
    print(f"\n🏆 {phase_name} Best: {best_score:.4f} ±{margin_of_error:.2f} (N={n_samples})")
    
    # Check for Statistical Significance (Lower bound > Benchmark)
    if (best_score - margin_of_error) > RULES_BASED_BENCHMARK:
        print(f"   ✅ Statistically Superior: {z_score:.1f} Standard Errors better than Rules-based Benchmark")
    else:
        print(f"   ⚠️ Not significantly better than Rules-based Benchmark ({RULES_BASED_BENCHMARK})")
    
    # 3. Save to Tracker
    progress_report.append({
        'Phase': phase_name,
        'Best_Score': best_score,
        'Margin_of_Error': margin_of_error,
        'N': n_samples
    })


    # Function 5:  Output parameter strings of the top 5 scoring bots from the phase
def output_top_5(df, phase_name):

    top_5_runs = df.sort_values(by='Avg_Final_Level', ascending=False).head(5)

    print("\n--- Top 5 Bots - Parameter Lists ---")
    print("=" * 40)

    #Iterate and print each parameter set
    for index, row in top_5_runs.iterrows():
    
        # Get the score for context
        score = row['Avg_Final_Level']
    
        # Create a list of the 10 parameter values from the row
        values_list = [row[param] for param in ALL_PARAMETERS]
    
        # Format all values as strings with high precision
        formatted_values = [f"{val:.4f}" for val in values_list]
    
        # Join them with ", "
        param_string = ", ".join(formatted_values)
    
        # Print the score and the parameter string
        print(f"{phase_name} Bot #{index} Score: {score:.2f}")
        print(param_string)
        print("-" * 40) # Separator


## PHASE 0 - Normalization and Calibration

Before beginning the optimization, we must calibrate the simulation engine. This phase performs two critical engineering tasks to ensure our experiment is valid:

**1. Input Normalization (Feature Scaling)**
The 10 optimization parameters operate on vastly different scales. If left unscaled, the optimizer would be biased toward the features with larger numbers. To fix this, I ran a preliminary logging pass and determined a specific High-Percentile Cutoff (98% - 99.5%) for each feature, tuning the threshold to exclude extreme outliers based on the distribution tail. These values became the Scale Factors. Dividing by these factors normalizes all inputs to a standard scale: 
* Scoring Parameters (e.g., `w_ratioGain`) are normalized to approximately $[0, 1]$.
* Bias Parameters (e.g., `w_attackBias`) are normalized to approximately $[-1, 1]$.

   
**2. Output Calibration (Noise Quantification)** 
The battleMage engine is stochastic, since it involves dice rolls for threat generation. Any given challenge outcome contains an element of luck. To  quantify the system's inherent noise, I ran a calibration set of 10,000 arena challenge simulations using my unoptimized champion. Here, we calculate the Standard Deviation ($\sigma$) of the Final Level reached. This $\sigma$ value allows us to calculate the Standard Error ($SE = \sigma / \sqrt{N}$) for any sample size. We will use this later to design safe cutoffs for "gates" that filter out bad bots early, and also to validate that our champion is significantly better than the Rules-based Benchmark.

In [ ]:
# PHASE 0: SYSTEM CALIBRATION
# =================================================================
# Before optimization, we must calibrate the engine to ensure inputs 
# are normalized and outputs are statistically reliable.
# =================================================================


# --- PART A: INPUT NORMALIZATION (Feature Scaling) ---
# The game features have vastly different ranges (e.g., Cost 0-100 vs Ratio 0-20,000).
# We run a logging pass to calculate Scale Factors (SF) to normalize them.

# 1. Load Data
ACTION_LOG_FILE = "data/phase_zero_actions.csv"
OUTCOME_LOG_FILE = "data/phase_zero_outcomes.csv"

try:
    actions_df = pd.read_csv(ACTION_LOG_FILE, low_memory=False)
    outcomes_df = pd.read_csv(OUTCOME_LOG_FILE)
    df = pd.merge(actions_df, outcomes_df, on="battle_id")
    
    # 2. Calculate 99th Percentile Scale Factors
    features_to_scale = {
        'f_cost': 'SF_COST',
        'f_ratioGain': 'SF_RATIO_GAIN',
        'f_ratioLoss': 'SF_RATIO_LOSS',
        'f_playerHPdelta': 'SF_PLAYER_HP_DELTA',
        'f_oppHPdelta': 'SF_OPP_HP_DELTA'
    }
    
    # Custom quantiles to handle outliers
    quantile_map = {
        'f_cost': 0.98, 'f_ratioGain': 0.99, 'f_ratioLoss': 0.99,
        'f_playerHPdelta': 0.995, 'f_oppHPdelta': 0.995
    }

    print("--- Calculated Scale Factors (Inputs) ---")
    for feature, name in features_to_scale.items():
        q = quantile_map[feature]
        scaler_value = df[feature].abs().quantile(q)
        if scaler_value == 0: scaler_value = 1.0
        print(f"{name:<20} : {scaler_value:.4f}")

except FileNotFoundError:
    print("WARNING: Phase 0 data not found. Skipping normalization calculation.")


# --- PART B: OUTPUT CALIBRATION (Noise Analysis) ---
# We ran 10,000 simulations to quantify the game's inherent variance (Sigma).
# This allows us to calculate "Safe Cutoffs" for our optimization funnel.

NOISE_FILE = "data/noise_calibration.csv"
try:
    df_noise = pd.read_csv(NOISE_FILE)
    raw_scores = df_noise['final_level']
    
    GLOBAL_SIGMA = raw_scores.std()
    CHAMPION_MEAN_BASELINE = raw_scores.mean()
    
    print(f"\n--- Game Engine Noise (Outputs) ---")
    print(f"Baseline Score: {CHAMPION_MEAN_BASELINE:.2f}")
    print(f"System Noise (σ): {GLOBAL_SIGMA:.4f}")


except FileNotFoundError:
    print("WARNING: Calibration data missing.")
    GLOBAL_SIGMA = 6.01 # Fallback

## PHASE 1 - Exploration

In phase one, we begin the search proper by performing a coarse-grained "brute force" mapping of the 10-dimensional parameter space. We want to find where in parameter space our highest-performing bots are found, and narrow the search space accordingly for the next phase.

**The Method:**
Our Java engine generated random values for each parameter within the normalized search space to create a candidate bot, and then ran the bot through a set of Arena Challenges. The Average Final Level reached was logged along with the values for each parameter.

* Volume: 1,000,000 bots were sampled uniformly from the normalized space.
* Precision: $N=100$ challenges per set. With a system noise ($\sigma$) of approximately 7 (calculated in Phase 0), we calculate a 95% Confidence Interval of $\pm 1.4$ levels. This is precise enough to distinguish capable bots from the rest, although not yet precise enough to distinguish between the very best.

We begin by loading our phase one data and generating a histogram of the Average Final Level Reached to visualize the distribution of performance. We'll also output the parameter strings of the top 5 bots, and check to see if we've met our goal of beating the Rules-based Benchmark.

In [ ]:
PHASE_1_NAME = "Phase 1"

#We adjust the quantile after viewing the distribution plots, aiming for a cutoff that gives enough bots to produce a meaningful signal,
#while excluding as many low-performing bots as possible. 
PHASE_1_QUANTILE = 0.9995

#we call our function to load phase 1 data and produce a histogram of the results
df_p1, top_runs_p1 = load_data_and_generate_histogram(
    file_path="data/phase_one_outcomes.csv", 
    phase_name=PHASE_1_NAME, 
    quantile_level=PHASE_1_QUANTILE)

output_top_5(df_p1, PHASE_1_NAME)
progress_update(df_p1, PHASE_1_NAME, n_samples=100)

### Phase 1 Results
We see immediately that almost all the bots perform poorly compared to our Rules-based Benchmark of Level 25.1. However, the tail of this plot is long, and we note that our elite cohort of the best 500 bots average a respectable Level 22.31. The absolute best bots in this sample technically scored higher than our benchmark, but given the high variance at $N=100$, this difference is not yet statistically significant.

We adjusted the quantile of the elite cohort after viewing the distribution plots, aiming for a cutoff that gives enough bots to produce a meaningful signal, while excluding as many lower-performing bots as possible. 

### Parameter Signal Analysis
The histogram proves that elite performance is rare but possible. To find it, we must narrow the search range of our parameters based on where we find the best performers. The following plots visualize the marginal distribution of each parameter for the Top 0.05% of runs (orange) compared to the full dataset (blue).

Any deviation from the uniform baseline indicates a signal. Sharp peaks or slopes in the orange distributions reveal the constraints of the champion strategy and will determine our choice of range in the next phase, while flat distributions indicate irrelevance and allow us to drop the parameter, reducing the dimensionality of our search.

In [ ]:
create_distribution_plots(df_p1, top_runs_p1, PHASE_1_NAME, PHASE_1_QUANTILE, INITIAL_EXPLORE_RANGES)

Our elite cohort shows clear trends, allowing us to categorize the parameters into four distinct signals:

* **Ramp Up (5 Parameters):** These plots show a clear preference for higher values. We will increase the lower bound to the **25th percentile** of the elite cohort, retaining 75% of the elite data points inside the new bounds.
* **Ramp Down (2 Parameters):** These plots show a preference for lower values. We will decrease the upper bound to the **75th percentile**, again retaining 75% of the elite data.
* **Flat/Complex (`w_alloc`):** This parameter, related to the allocation of resources between Stamina and HP, shows a bimodal peak. We categorize this as "Complex" which will retain the entire range for the next search phase, but choose to clip the very bottom of the range by hand as there is a clear region of no signal there.
* **Removed (`w_cost`):** By comparison, this parameter is flat across its entire range. As a penalty for the resource cost of an action, its irrelevance is likely because the cost is already indirectly captured by the other delta parameters. We mark this as "Removed" and will set it to 0.0.

Methodology Note: We select the **75% retention threshold** (25th/75th percentiles) as a conservative filter. This narrows the search space significantly while minimizing the risk of missing the region containing the global best.

Based on our categorizations, we now calculate each parameter's search ranges for the next phase.

In [ ]:
P1_CATEGORIES = {'Ramp Up': ['w_ratioGain', 'w_playerHPdelta','w_oppHPdelta','w_attackBias','w_blockBias','w_blastBias'],
                 'Ramp Down': ['w_ratioLoss','w_shieldBias'],
                 'Flat/Complex': ['w_alloc'],
                 'Removed':['w_cost']}

ranges_for_p1r = calculate_new_ranges(top_runs_p1, INITIAL_EXPLORE_RANGES, P1_CATEGORIES)

ranges_for_p1r['w_alloc'] = (0.2, 1.0)
print("clipped w_alloc by hand to (0.2, 1.0)")

## PHASE 1 REFINED - Exploration

We continue exploring in our narrowed search space. We've reduced the dimensionality by one. And as we'll calculate below, even with our modest reductions in each parameter's range, we've reduced the 9-D volume of the space by 99.5% 

* Volume: 2,000,000 bots were sampled uniformly from the narrowed range.
* Precision: $N=1000$ challenges per *promising** set. With a system noise of approximately 7, we calculate a 95% Confidence Interval of $\pm 0.44$
 levels.

*Here we determine that too much time is spent running battles with clearly bad bots and introduce a system of **performance gates** to save time. After 10 challenges, only bots that have scored better than level 10 are allowed to continue. After 100 challenges, only bots that are better than level 22 continue to run the full set of $N=1000$ challenges. This idea is developed further for phase two, see below.

We load our new data, generate the performance histogram, and output our winning bots:

In [ ]:
PHASE_1R_NAME = "Phase 1 Refined"
PHASE_1R_QUANTILE = 0.99999

df_p1r, top_runs_p1r = load_data_and_generate_histogram(
    file_path="data/phase_one_refined_outcomes.csv", 
    phase_name=PHASE_1R_NAME, 
    quantile_level=PHASE_1R_QUANTILE)

output_top_5(df_p1r, PHASE_1R_NAME)
progress_update(df_p1r, PHASE_1R_NAME, n_samples=1000)

### Phase 1 Refined: Results
The histogram shows a dramatic improvement over Phase 1. The "immediate failure" rate has dropped significantly, and the high-performance tail is much thicker. Most importantly, **the best bots in this phase have already met our primary goal, surpassing the Rules-based Benchmark (25.1) by more than 4 Standard Errors**.

### Parameter Signal Analysis
We find after viewing the next set of distribution plots that an ultra-elite quantile of the top 0.001%, or just the **Top 20 runs** is necessary to find the signal that will enable us to "zero in" on the exact parameter ranges for the final phase.

In [ ]:
create_distribution_plots(df_p1r, top_runs_p1r, PHASE_1R_NAME, PHASE_1R_QUANTILE, ranges_for_p1r)

The distribution plots for the refined cohort reveal sharper trends, allowing us to categorize the remaining 9 parameters into two distinct groups for the next phase:

* **"Ramp Up" (3 Parameters):** These plots (`w_ratioGain`, `w_ratioLoss`, `w_playerHPdelta`) still show a directional ramp rather than a centered peak. We will continue to use a **75% retention threshold** (cutting the bottom quartile) to narrow these ranges further without cutting off the signal.
* **"2SD" (6 Parameters):** These plots now show clear, bell-curve style peaks. For these, we will switch to a **Mean ± 2 Standard Deviations** method. This dramatically tightens the search range around the "sweet spot," and we'll clip the range to ensure it does not expand beyond our current boundaries.

Based on these categorizations, we will now calculate the search ranges to use in Phase 2.

In [ ]:
P1R_CATEGORIES = {
    'Ramp Up': ['w_ratioGain', 'w_ratioLoss', 'w_playerHPdelta'],
    '2SD': ['w_alloc', 'w_oppHPdelta', 'w_attackBias', 'w_blockBias', 
            'w_blastBias', 'w_shieldBias'],
    'Removed': ['w_cost']
}

ranges_for_p2 = calculate_new_ranges(top_runs_p1r, ranges_for_p1r, P1R_CATEGORIES)

ranges_for_p2['w_attackBias'] = (-0.364, 0.601) 
ranges_for_p2['w_blockBias'] = (-0.433, 0.573)
print("***used unclipped ranges for attackBias(-0.364, 0.601) and blockBias(-0.433, 0.573)")

## Optimization Strategy: Designing the Progressive Gating System

The Problem:
Phase 1 proved that while we could find "good" bots, at $N=1000$ the system is still too noisy to crown a confirmed champion. We'd like a precise final score for our champion and so we plan to run the best bots to N=500,000, which will give a standard error of about 0.01 Level.

Running 2 million candidates at $N=500,000$ would require one trillion simulations, which is computationally unfeasible.

The Solution: Iterative Gating
To solve this, I engineered a progressive filtering system that evolved over the two phases:

1.  Phase 1 Refined: I implemented a simple 2-stage gate capped at $N=1,000$. This showed we could safely discard most bots early without losing signal.
2.  Phase 2: I designed a robust 5-Stage Progressive Gating System capped at **$N=500,000$**.

The Dynamic Logic:
The Phase 2 engine is self-optimizing. It assumes the current "Champion" is the benchmark.
* Fail Fast: Bots are checked at $N=10, 100, 1000, 10000,$ and $50000$.
* Safety Margin: Cutoffs are set to **3.8 Standard Errors (SE)** below the benchmark, ensuring a $<0.01\%$ chance of accidental dismissal.
* Dynamic Updates: Whenever a new Champion is verified (at $N=500,000$), the engine automatically raises the bar, recalculating all cutoffs to be 3.8 SE below the *new* gold standard.

Now we give sample cutoff level calculations, using the Rules-based benchmark as standard.

In [ ]:


# =================================================================
# 3. OPTIMIZATION DESIGN: CALIBRATING THE FUNNEL
# =================================================================
# Before running the massive Phase 2 search, we needed to determine
# statistically safe "cutoffs" to filter bad bots early without 
# accidentally discarding a champion.
# =================================================================


print(f"--- Gating Calibration ---")
print(f"Benchmark Score: {RULES_BASED_BENCHMARK:.4f}")
print(f"System Noise (σ): {GLOBAL_SIGMA:.4f}")
print("-" * 60)

# 3. Define Safety Margins (3-Sigma Rule)
# We calculate the cutoff for each gate size (N).
# Any bot scoring below this limit is >99.7% likely to be inferior.
print(f"{'Gate (N)':<10} | {'Std Error':<10} | {'Margin (3.8σ)':<12} | {'SAFE CUTOFF'}")
print("-" * 60)
    
gate_Ns = [100, 1000, 10000, 50000]
    
for n in gate_Ns:
    se = GLOBAL_SIGMA / np.sqrt(n)  # Standard Error
    margin = 3.8 * se                     # 3-Sigma Safety Margin
    cutoff = CHAMPION_MEAN_BASELINE - margin     # The Gate
        
    print(f"{n:<10} | {se:<10.3f} | {margin:<12.3f} | > {cutoff:.2f}")



## PHASE 2 TRIAL - Exploitation

Using the ranges we determined in the refined Phase 1, our initial search volume has been narrowed by **99.997%** (see calculation below). To make the remaining computational load feasible, we deployed our Progressive Gating System. Armed with this focused search space and high-efficiency engine, we are now fully equipped to discover the "True Champion."

After a trial run of 50,000 bots, we identified a signal clear enough to narrow our search range by another **96.5%** before embarking on our final "Phase 2 Refined" run.

Now, we load and display the data from this Phase 2 trial run.

In [ ]:
PHASE_2_NAME = "Phase 2 Trial"
PHASE_2_QUANTILE = 0.9995

df_p2, top_runs_p2 = load_data_and_generate_histogram(
    file_path="data/phase_two_outcomes.csv", 
    phase_name=PHASE_2_NAME, 
    quantile_level=PHASE_2_QUANTILE)

output_top_5(df_p2, PHASE_2_NAME)
progress_update(df_p2, PHASE_2_NAME, n_samples=500000)

### Phase 2 Trial: Results
The results again show a dramatic improvement over the previous phase. The "immediate failure" rate has nearly vanished, and a sizable fraction of the population now performs at a high level. Most importantly, the best bot in this trial run has already surpassed the Rule-Based Benchmark by **61 Standard Errors**.

### Parameter Signal Analysis (Phase 2 Trial)
The parameter distributions for the **Top 0.05%** of this trial run reveal clear signals that allow us to drastically shrink the search volume again—achieving a total reduction of **99.999908%** from the original search space—before launching the final refined phase 2 run.

In [ ]:
create_distribution_plots(df_p2, top_runs_p2, PHASE_2_NAME, PHASE_2_QUANTILE, ranges_for_p2)

We can classify these plots into two distinct categories:

* **Robust (4 Parameters):** These plots (`w_ratioGain`, `w_ratioLoss`, `w_playerHPdelta`, `w_oppHPdelta`) show a broad "plateau" where elite bots exist across most or all of the current search range. This indicates that while the parameter is important, the specific value is flexible. We categorize these as "Flat/Complex" and will retain their full range in the final phase to allow the optimizer to find complex interactions.
* **Sensitive (5 Parameters):** These plots (`w_alloc` and all 4 Biases) show a sharp, narrow peak. The champion strategy requires these values to be tuned very precisely. We categorize these as "2SD" and will narrow the search range to the Mean ± 2 Standard Deviations to zoom in on the "sweet spot."

Based on these signals, we calculate the final "Ultra-Tight" ranges for the definitive Phase 2 Refined run.

In [ ]:
P2_CATEGORIES = {
    '2SD': ['w_alloc', 'w_attackBias', 'w_blockBias', 
                         'w_blastBias', 'w_shieldBias'], #the sensitive parameters
    'Flat/Complex': ['w_ratioGain', 'w_ratioLoss', 'w_playerHPdelta', 
                         'w_oppHPdelta'], #The robust parameteres
    'Removed':['w_cost']
}

ranges_for_p2r = calculate_new_ranges(top_runs_p2, ranges_for_p2, P2_CATEGORIES)

## PHASE 2 REFINED
This is the final phase of our search. The trial run of Phase 2 found a Level 25.7 Champion. The initial gold standard for our progressive gate is set to this value.
* Volume: 1,000,000 bots
* Precision: $N=500,000$ for champion contenders

Now we load our final dataset, generate the results histogram, and output our Champion bot:

In [ ]:
PHASE_2R_NAME = "Phase 2 Refined"
PHASE_2R_QUANTILE = 0.9999

df_p2r, top_runs_p2r = load_data_and_generate_histogram(
    file_path="data/phase_two_refined_outcomes.csv", 
    phase_name=PHASE_2R_NAME, 
    quantile_level=PHASE_2R_QUANTILE)

output_top_5(df_p2r, PHASE_2R_NAME)
progress_update(df_p2r, PHASE_2R_NAME, n_samples=500000)

### Phase 2 Refined: Verification & Convergence

The final histogram reveals a complete transformation of the agent population. Unlike Phase 1, where elite performance was a statistical anomaly, here it is the expected standard. Comparing this plot to that of the phase 2 trial run, we see that we have successfully removed the larger, left peak of lower-performing bots, and have found an efficient search space for this final run.

* **Convergence:** The distribution has shifted entirely to the right. The "tail" of incompetent bots has been eliminated.
* **The Hard Cap:** The distribution piles up sharply against the Level 26-27 range, suggesting that we are approaching a "Hard Cap" on the possible final level achieved, resulting from Threat scaling. 
* **Stability:** With $N=500,000$, the Standard Error is negligible ($\pm 0.01$). We are confident we have found the location of the global maximum.

### Conclusion: The Optimization Trajectory

To visualize the results of this engineering challenge, we plot the full trajectory of the champion across all four phases. This plot combines our performance metrics (Average Final Level) with our uncertainty metrics (95% Confidence Intervals) to demonstrate how the process systematically improved the quality of generated bots.

In [ ]:

# --- Setup Data (Reloading for context) ---
# summary_df should have: 'Phase', 'Best_Score', 'Margin_of_Error', 'N'
summary_df = pd.DataFrame(progress_report)
# Calculate Stats
summary_df['Std_Error'] = GLOBAL_SIGMA / np.sqrt(summary_df['N'])
summary_df['Margin_of_Error'] = 1.96 * summary_df['Std_Error']
summary_df['Z_Score'] = (summary_df['Best_Score'] - RULES_BASED_BENCHMARK) / summary_df['Std_Error']

# Label Logic
summary_df['Label'] = summary_df.apply(lambda x: f"{x['Phase']}\n(N={int(x['N']):,})", axis=1)

# --- Plotting ---
plt.figure(figsize=(12, 7))
sns.set_style("whitegrid")

# Dynamic Limits
min_y = (summary_df['Best_Score'] - summary_df['Margin_of_Error']).min()
max_y = (summary_df['Best_Score'] + summary_df['Margin_of_Error']).max()
y_lower = min(min_y, RULES_BASED_BENCHMARK) - 0.5
y_upper = max_y + 0.3

# 1. Benchmark Line
plt.axhline(RULES_BASED_BENCHMARK, color='crimson', linestyle='--', linewidth=2, alpha=0.8, 
            label=f"Rule-Based Benchmark ({RULES_BASED_BENCHMARK})")
plt.fill_between(
    [-0.5, len(summary_df) - 0.5], 
    20, RULES_BASED_BENCHMARK, 
    color='crimson', alpha=0.05
)

# 2. Error Bars & Line
plt.errorbar(
    x=summary_df['Label'], 
    y=summary_df['Best_Score'], 
    yerr=summary_df['Margin_of_Error'], 
    fmt='o-', 
    markersize=10, 
    linewidth=2, 
    capsize=10, 
    color='#2c3e50',
    label='Phase-Best Score (95% CI)'
)

# 3. The "Z-Score" Annotation
for i, row in summary_df.iterrows():
    z = row['Z_Score']
    if z < 1.96:
        text_label = "Not Sig."
        color = 'gray'
        fontweight = 'normal'
    else:
        text_label = f"+{z:.1f} SE" 
        color = 'green'
        fontweight = 'bold'
    
    top_of_bar = row['Best_Score'] + row['Margin_of_Error']
    plt.text(
        i, 
        top_of_bar + 0.05, 
        text_label, 
        ha='center', 
        va='bottom', 
        color=color, 
        fontweight=fontweight,
        fontsize=11
    )

plt.title("Optimization Trajectory: Phase-Best Bots", fontsize=16, fontweight='bold')
plt.ylabel("Average Final Level", fontsize=12)
plt.xlabel(None)
plt.ylim(y_lower, y_upper)
plt.grid(axis='x', alpha=0)
plt.legend(loc='upper right')

plt.tight_layout()
plt.savefig("images/optimization_trajectory.png", dpi=300)
plt.show()

## Next Steps: Strategic Analysis
This trajectory proves we have successfully engineered a champion ($\text{Avg Level: 26.07}$) that is statistically superior to the Rule-Based Benchmark. We have solved the optimization problem. However, we are interested in studying further the strategy that the champion bot uses, especially as compared to our Rules-based benchmark. In Notebook 2: Strategic Analysis, we will continue this study, using Machine Learning to reverse-engineer the champion's strategy and gain insight into *why* it outperforms the hand-coded benchmark.

## Appendices

### Appendix A: Parameter Distribution plots for Refined Phase Two
We present here the distribution plots for the final run. We see some sharp peaks, and could profitably narrow our search volume further, continuing our optimization process. But we must stop sometime, and we beleive we are very close to the game's theoretical limit at this point, evidenced by the final histogram ending in a "cliff" rather than a "tail."


In [ ]:
create_distribution_plots(df_p2r, top_runs_p2r, PHASE_2R_NAME, PHASE_2R_QUANTILE, ranges_for_p2r)

### Appendix B: Search Space Reduction

We calculate the reduction in the hyperspace search volume in each phase, compared to the previous phase, and to the original search range. We ignore the zero-sized range of the removed parameter.

In [ ]:
PHASE_RANGES = [
    ("Phase 1", INITIAL_EXPLORE_RANGES),
    ("P1 Refined", ranges_for_p1r),
    ("Phase 2", ranges_for_p2),
    ("P2 Refined", ranges_for_p2r)
]


def calculate_volume(ranges: Dict[str, Tuple[float, float]]) -> Tuple[float, int]:
    """Calculates the geometric volume and the number of active dimensions."""
    volume = 1.0
    active_dims = 0
    
    for param, (min_val, max_val) in ranges.items():
        # Only count dimensions where the range size is non-zero
        range_size = max_val - min_val
        if range_size > 1e-6: # Check if size is greater than a tiny epsilon
            volume *= range_size
            active_dims += 1
            
    return volume, active_dims


def generate_volume_table(phase_data: List[Tuple[str, Dict]]) -> pd.DataFrame:
    """
    Generates a DataFrame showing dimension count and volume reduction across phases.
    """
    results = []
    previous_volume = 1.0 # Start with the normalized initial volume
    initial_volume = 1.0
    
    # Calculate the Initial Volume from the 10-D space for accurate Total Reduction
    # (1.0 for [0, 1] range, 2.0 for [-1, 1] range)
    initial_volume = calculate_volume(INITIAL_EXPLORE_RANGES)[0]
    
    for i, (name, ranges) in enumerate(phase_data):
        current_volume, active_dims = calculate_volume(ranges)
        
        # Calculate reduction from the previous phase (Phase Reduction)
        if i == 0:
            # The first step is the "total" 10-D volume
            reduction_from_prev = 0.0 
        else:
            reduction_from_prev = 1.0 - (current_volume / previous_volume)
        
        # Calculate reduction from the initial (Phase 1) volume (Total Reduction)
        reduction_from_initial = 1.0 - (current_volume / initial_volume)
        
        results.append({
            'Phase': name,
            'Dimensions': active_dims,
            'Vol_Remaining': current_volume,
            'Phase Reduction': f"{(reduction_from_prev * 100):.4f}%",
            'Total Reduction': f"{(reduction_from_initial * 100):.6f}%"
        })
        
        # Update the baseline for the next phase
        previous_volume = current_volume
        
    # Final formatting of the DataFrame
    df_results = pd.DataFrame(results)
    
    # Hide the "Total Reduction" for the first step as it is the baseline.
    df_results.loc[0, 'Phase Reduction'] = 'N/A (Baseline)'
    df_results.loc[0, 'Total Reduction'] = 'N/A (Baseline)'
    
    return df_results[['Phase', 'Dimensions', 'Phase Reduction', 'Total Reduction']]


# --- Execution ---
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.6f}'.format)

# Generate the final table
summary_table = generate_volume_table(PHASE_RANGES)

print("\n\n📊 Optimizatyion Process Analysis: Search Space Reduction")
print("---------------------------------------------------------")
print(summary_table.to_markdown(index=False))

### Appendix C - Source Code Links

All Java source code is included in this repository. Interesting classes are:

* [**The Bot:**](BotBattler/src/optimization/OptimizingChampion.java) - The 10-parameter decision engine.

* [**The Optimizer:**](BotBattler/src/optimization/Optimizer_Phase_Two.java) - The Monte Carlo code and gating system implementation.

* [**The Simulation:**](BotBattler/src/game/Battle.java) - The stochastic battle engine.

### Appendix D - Results Histograms Presented Side-by-Side

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

plots = [
    (df_p1,  "Phase 1 — Broad Exploration\n1,000,000 parameter sets"),
    (df_p1r, "Phase 1 Refined — Focused Exploration\n2,000,000 parameter sets"),
    (df_p2,  "Phase 2 — Exploitation (Trial)\n52,000 parameter sets"),
    (df_p2r, "Phase 2 Refined — Exploitation\n1,005,000 parameter sets")
]

for ax, (df, title) in zip(axes.flat, plots):
    sns.histplot(
        data=df,
        x="Avg_Final_Level",
        bins=50,
        kde=True,
        stat="probability",
        common_norm=False,
        ax=ax
    )
    ax.set_title(title, fontsize=14, fontweight="semibold")
    ax.set_xlabel("Average Final Level Achieved")
    ax.set_ylabel("Fraction of Parameter Sets")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_ylim(0, 0.08)


plt.tight_layout()
plt.savefig("images/battleMage_phase_distributions_2x2.png", dpi=200)
plt.show()
